# UGA Grad Survivor — Monte Carlo Simulation
Reproduces all game mechanics (stats, perks, passive drains, milestones, card pools) and runs thousands of games to analyze balance across archetype × PI combinations.

In [ ]:
import random
import math
import copy
from collections import defaultdict, Counter
import pandas as pd
import numpy as np

# Try optional viz libs
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid')
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

## 1. Game Data

In [ ]:
# ── Archetypes ──
ARCHETYPE_DATA = {
    'overachiever':   {'st': {'mind':50,'body':45,'wallet':60,'bonds':50,'research':55}},
    'vibe_coder':     {'st': {'mind':55,'body':40,'wallet':60,'bonds':45,'research':60}},
    'fun_haver':      {'st': {'mind':40,'body':55,'wallet':50,'bonds':70,'research':35}},
    'global_student': {'st': {'mind':50,'body':50,'wallet':60,'bonds':55,'research':50}},
    'biologist':      {'st': {'mind':45,'body':50,'wallet':55,'bonds':60,'research':40}},
    'double_agent':   {'st': {'mind':50,'body':45,'wallet':55,'bonds':50,'research':55}},
    'gym_bro':        {'st': {'mind':40,'body':70,'wallet':35,'bonds':55,'research':45}},
    'neurodivergent': {'st': {'mind':55,'body':45,'wallet':50,'bonds':50,'research':50}},
}

# ── PI Types ──
PI_TYPES = ['micromanager','ghost','exploiter','mentor','new_pi','dynasty']

In [ ]:
# ── Card definitions ──
# Each card: id, tag, eL, eR, optional: minSem, maxSem, exclusive, requires, sets, milestone, semester, piSelection
# We only need the mechanical fields for simulation.

def c(id, tag='', eL=None, eR=None, minSem=None, maxSem=None,
      exclusive=None, requires=None, sets=None,
      milestone=False, semester=None, piSelection=False):
    return {
        'id': id, 'tag': tag,
        'eL': eL or {}, 'eR': eR or {},
        'minSem': minSem, 'maxSem': maxSem,
        'exclusive': exclusive, 'requires': requires, 'sets': sets,
        'milestone': milestone, 'semester': semester, 'piSelection': piSelection,
    }

# ── PHASE 1 CARDS (25) ──
PHASE1_CARDS = [
    c('orientation','Day One', eL={'mind':5,'bonds':10,'network':5}, eR={'mind':10,'bonds':5}),
    c('first_hpc','First Day', eL={'bonds':10,'mind':5,'research':5}, eR={'mind':5,'body':-5,'research':5}, sets=['asked_senior']),
    c('first_rotation','Rotation', eL={'bonds':-10,'wallet':15,'mind':10,'research':5}, eR={'bonds':20,'mind':10,'wallet':-5,'research':5}),
    c('first_ta','Teaching', eL={'mind':10,'body':-10,'research':-5}, eR={'mind':-10,'body':5}),
    c('first_conf_poster','Conference', eL={'bonds':15,'mind':-10,'research':5,'network':10}, eR={'mind':5,'bonds':5,'research':5,'network':5}),
    c('cohort_party','Social', eL={'bonds':20,'mind':15,'body':-5,'research':-5}, eR={'mind':5,'bonds':-5,'research':5}),
    c('imposter_moment','Existential Event', eL={'mind':10,'bonds':5,'research':5}, eR={'mind':5,'bonds':-5,'research':5}),
    c('first_pipeline','Pipeline', eL={'mind':-10,'bonds':5,'research':10}, eR={'mind':5,'bonds':-5,'research':5}, sets=['built_pipeline']),
    c('early_burnout','Health', eL={'body':15,'mind':15,'bonds':5,'research':-5}, eR={'body':-10,'mind':-5,'research':5}),
    c('first_review','Milestone', eL={'bonds':10,'mind':5,'research':5}, eR={'bonds':5,'mind':-10,'research':5}),
    c('quals_announced','Milestone', eL={'mind':-15,'body':-10,'bonds':-5,'research':10}, eR={'mind':5,'bonds':5,'research':-5}, sets=['quals_scheduled']),
    c('orientation_bbq','Social', eL={'bonds':15,'mind':-5,'research':5}, eR={'body':10,'bonds':5}),
    c('lab_philosophy','Advisor', eL={'bonds':10,'mind':-5,'research':5}, eR={'mind':5,'bonds':-5,'research':5}),
    c('stipend_letter','Day One', eL={'wallet':10,'bonds':-5}, eR={'mind':-5,'wallet':5}),
    c('campus_bus','Athens Life', eL={'wallet':-8,'body':10,'mind':5}, eR={'body':-10,'mind':-5,'wallet':5}),
    c('farmers_market','Athens Life', eL={'body':10,'mind':-5,'wallet':-5}, eR={'mind':10,'body':-5,'wallet':-5}),
    c('first_gen','Privilege', eL={'bonds':15,'mind':5,'network':10}, eR={'mind':5,'bonds':-10,'research':5}),
    c('partner_subsidy','Privilege', eL={'wallet':5,'bonds':10,'mind':-10}, eR={'mind':-5,'wallet':5}),
    c('unwritten_rules','Privilege', eL={'bonds':15,'mind':-5,'network':10}, eR={'mind':5,'bonds':-10}),
]

# ── PHASE 2 CARDS (inline, 48) ──
PHASE2_CARDS = [
    c('advisor_ghost','Advisor', eL={'bonds':-15,'mind':-5,'research':-5}, eR={'mind':-15,'bonds':-5,'research':-5}),
    c('meeting_cancel','Advisor', eL={'bonds':5,'mind':-15,'research':-5}, eR={'bonds':-5,'mind':10}),
    c('advisor_rewrite','Advisor', eL={'mind':-15,'bonds':10,'research':5}, eR={'bonds':-5,'mind':10,'research':5}),
    c('conf_inspiration','Advisor', eL={'mind':-20,'bonds':15,'research':5}, eR={'bonds':-10,'mind':10}),
    c('career_advice','Advisor', eL={'mind':-15,'bonds':-5}, eR={'bonds':5,'mind':-5}),
    c('sapelo_down','Pipeline Failure', eL={'research':5,'mind':5}, eR={'mind':-5}),
    c('conda_hell','Pipeline Failure', eL={'mind':5,'research':5}, eR={'mind':-10,'body':-10,'research':5}),
    c('p_value','Pipeline Failure', eL={'mind':-15,'bonds':5,'research':5}, eR={'mind':-10,'bonds':-5,'research':10}),
    c('lab_fridge','Lab Politics', eL={'mind':5,'bonds':-15}, eR={'mind':15,'bonds':-10,'body':-5}),
    c('lab_meeting','Lab Politics', eL={'mind':-20,'body':-15,'research':5}, eR={'mind':-10,'bonds':-15}, sets=['pulled_allnighter']),
    c('rotation_student','Lab Politics', eL={'bonds':15,'mind':-10,'research':5}, eR={'mind':5,'bonds':-5,'research':5}, sets=['mentored_student']),
    c('not_eaten','Human Existence', eL={'body':-10,'mind':5}, eR={'wallet':-8,'body':5,'mind':10}),
    c('friend_text','Human Existence', eL={'bonds':10,'mind':-10}, eR={'bonds':20,'mind':-5}),
    c('pickleball','Human Existence', eL={'body':20,'bonds':15,'mind':10,'research':-5}, eR={'mind':-15,'body':-10,'wallet':10,'research':5}),
    c('stipend_late','Funding', eL={'wallet':15,'bonds':-5,'mind':-5}, eR={'wallet':-10,'mind':15}),
    c('grant_deadline','Funding', eL={'wallet':20,'mind':-20,'body':-15,'research':10}, eR={'mind':5,'body':5}, sets=['applied_grant']),
    c('linkedin','Existential Event', eL={'mind':-15,'wallet':5,'network':10}, eR={'mind':5,'body':10}),
    c('is_publishable','Existential Event', eL={'mind':-20,'bonds':-10}, eR={'mind':-5,'bonds':5,'research':10}),
    c('pipeline_runs','Rare Win', eL={'mind':20,'bonds':15,'body':10,'research':-5}, eR={'mind':5,'bonds':-5,'research':10}),
    c('nih_freeze','Funding Crisis', eL={'wallet':5,'mind':-15,'bonds':-5}, eR={'mind':-10,'bonds':5}),
    c('rent_hike_v2','Athens Reality', eL={'wallet':5,'bonds':10,'mind':-5}, eR={'wallet':-8,'mind':-5,'body':5}),
    c('international_student','Lab Life', eL={'bonds':10,'mind':-10}, eR={'bonds':15,'mind':-5}),
    c('nsf_cut','Funding Crisis', eL={'bonds':10,'mind':-10,'wallet':-5}, eR={'mind':-15,'wallet':5}),
    c('side_gig','Temptation', eL={'wallet':20,'mind':-10,'bonds':-10,'research':5}, eR={'wallet':-5,'mind':5,'bonds':5}),
    c('game_day','Athens Life', eL={'bonds':15,'body':5,'mind':-10,'research':-5}, eR={'mind':10,'bonds':-5,'research':5}),
    c('someone_submitted','Lab Politics', eL={'bonds':20,'mind':-10,'research':-5}, eR={'mind':-15,'bonds':-10,'research':5}),
    c('reviewer_2','Peer Review', eL={'mind':-20,'body':-10,'research':10}, eR={'mind':-10,'bonds':-5,'research':5}),
    c('conf_accepted','Rare Win', eL={'mind':20,'bonds':10,'research':10,'network':5}, eR={'bonds':20,'wallet':-10,'mind':10,'research':5,'network':10}),
    c('reviewer_1','Rare Win', eL={'mind':15,'research':5}, eR={'mind':-10,'bonds':10,'research':10}),
    c('github_down','Pipeline Failure', eL={'mind':5,'research':5}, eR={'mind':-5,'body':-5}),
    c('kallisto','Pipeline Failure', eL={'mind':-15,'bonds':15,'research':10}, eR={'mind':-5,'bonds':-20,'research':-5}),
    c('scratch_purge','Pipeline Failure', eL={'research':-5,'mind':5}, eR={'body':-15,'research':-10,'mind':-10}),
    c('job_pending','Pipeline Failure', eL={'research':5,'mind':5}, eR={'mind':-5}),
    c('academic_twitter','Marketing', eL={'bonds':10,'mind':-15,'body':-5,'network':20}, eR={'mind':5,'bonds':-10,'network':10}),
    c('personal_website','Marketing', eL={'mind':-10,'wallet':5,'network':15}, eR={'mind':5,'bonds':-5,'network':10}),
    c('graphical_abstract','Marketing', eL={'mind':-15,'bonds':10,'research':5}, eR={'mind':5,'bonds':-5,'research':5}),
    c('ta_overload','Teaching Labor', eL={'bonds':-10,'mind':10,'body':5,'research':5}, eR={'body':-15,'mind':-10,'wallet':10,'research':-5}),
    c('student_crisis','Teaching Labor', eL={'bonds':10,'mind':-15,'body':-10}, eR={'mind':-5,'bonds':-5}),
    c('teaching_doesnt_count','Teaching Labor', eL={'bonds':15,'mind':10,'wallet':-5,'research':-5}, eR={'mind':5,'bonds':-5,'wallet':5,'research':10}),
    c('therapy_waitlist','Mental Health', eL={'mind':10,'bonds':5}, eR={'mind':15,'wallet':-8}),
    c('comparison_spiral','Existential Event', eL={'mind':10,'body':10,'bonds':5,'research':-5}, eR={'mind':-10,'wallet':5,'research':5}),
    c('medication_cost','Mental Health', eL={'mind':15,'wallet':-10,'body':5}, eR={'mind':-10,'body':-5,'wallet':5}),
    c('authorship_politics','Privilege', eL={'bonds':-15,'mind':15,'research':5}, eR={'mind':-15,'bonds':5,'research':5}),
    c('unpaid_summer','Privilege', eL={'wallet':10,'mind':-15,'bonds':5,'research':5}, eR={'wallet':15,'mind':5,'bonds':-10,'research':-5}),
    c('gender_dynamics','Privilege', eL={'mind':10,'bonds':-10,'research':5}, eR={'mind':-15,'bonds':5,'research':5}),
    c('car_accident','Life Event', eL={'body':10,'mind':5,'wallet':-10,'bonds':5,'research':-5}, eR={'body':-10,'mind':-5,'wallet':-8,'bonds':10,'research':-5}),
    c('family_emergency','Life Event', eL={'bonds':15,'wallet':-10,'mind':-10,'research':-5}, eR={'mind':-10,'bonds':-5,'wallet':5}),
    c('pandemic_echo','Black Swan', eL={'mind':5,'bonds':-10,'body':10,'research':5}, eR={'body':-15,'mind':-5,'bonds':5,'research':5}),
    c('federal_shutdown','Black Swan', eL={'wallet':5,'mind':-10,'body':-5}, eR={'mind':-5,'wallet':-10}),
    c('housing_crisis','Black Swan', eL={'bonds':15,'wallet':-5,'mind':-10}, eR={'body':-5,'wallet':-5,'mind':5}),
    c('lab_flood','Black Swan', eL={'mind':-15,'body':-10,'bonds':10,'research':5}, eR={'mind':5,'body':10,'bonds':-5,'research':-5}),
    c('war_funding','Black Swan', eL={'wallet':5,'mind':-15,'bonds':-5}, eR={'wallet':-10,'mind':-10}),
    c('no_sick_days','Athens Reality', eL={'body':15,'mind':10,'bonds':-10,'research':-5}, eR={'body':-10,'mind':-5,'bonds':10,'research':5}),
    c('dental_emergency','Life Event', eL={'wallet':-8,'body':10,'mind':-5}, eR={'wallet':-5,'body':-5,'mind':-10}),
    c('recommendation_favor','Academic Politics', eL={'bonds':15,'body':-15,'mind':-5,'research':-5,'network':5}, eR={'mind':10,'bonds':-10,'research':5}),
    c('advisor_poached','Cruel', eL={'bonds':-20,'mind':10,'research':5}, eR={'mind':-15,'bonds':-10,'research':5}),
    c('scooped','Cruel', eL={'mind':-20,'bonds':-5,'body':-10,'research':10}, eR={'mind':-15,'bonds':5,'research':10}),
    c('cohort_dropout','Cruel', eL={'bonds':5,'mind':-20}, eR={'mind':-10,'bonds':-15,'research':10}),
    c('funding_terminated','Cruel', eL={'mind':-15,'wallet':5,'body':-10,'research':5}, eR={'wallet':10,'mind':-10,'bonds':-10,'research':-5}),
    c('data_breach','Cruel', eL={'bonds':-10,'mind':-15,'body':-10,'research':5}, eR={'mind':-20,'body':-5,'research':5}),
    c('retaliation','Cruel', eL={'bonds':-20,'mind':5,'wallet':-5,'research':-10}, eR={'mind':-15,'bonds':-10,'research':-5}),
    c('grade_dispute_escalation','Cruel', eL={'mind':-10,'bonds':-15,'research':-5}, eR={'mind':-15,'bonds':-5,'body':-5,'research':-5}),
    c('imposter_vindicated','Cruel', minSem=8, eL={'bonds':-10,'mind':-5,'body':5,'research':5}, eR={'body':-20,'mind':-10,'wallet':-5,'research':10}),
    c('stress_baking','Big Mood', eL={'mind':20,'body':-5,'bonds':10,'research':-5}, eR={'mind':-10,'body':-5,'wallet':5,'research':5}, sets=['baked_cookies']),
]

# ── NEW_PHASE2_CARDS (from js/cards-new-phase2.js, 51) ──
NEW_PHASE2_CARDS = [
    c('advisor_weekend_email','Advisor', eL={'research':5,'mind':-15,'body':-10,'bonds':5}, eR={'mind':5,'bonds':-15}),
    c('advisor_takes_credit','Advisor', eL={'bonds':-15,'mind':10}, eR={'mind':-10,'bonds':-5}),
    c('advisor_gaslighting','Advisor', eL={'bonds':-15,'mind':10}, eR={'mind':-15,'research':-5}),
    c('advisor_favorite','Advisor', eL={'bonds':-10,'mind':5}, eR={'mind':-5,'research':10,'body':-10}),
    c('advisor_moving_goalposts','Advisor', eL={'bonds':-10,'mind':10}, eR={'mind':-15,'body':-10,'research':5}),
    c('advisor_vacation','Advisor', eL={'bonds':-5,'mind':-5}, eR={'mind':-10,'research':5,'bonds':-5}),
    c('advisor_onedrive','Advisor', eL={'mind':-5,'research':5,'bonds':5}, eR={'mind':-10,'body':-5}),
    c('not_in_group_chat','Lab Drama', eL={'bonds':10,'mind':-15}, eR={'mind':-10,'bonds':-10}),
    c('lab_gossip','Lab Drama', eL={'bonds':-10,'mind':5}, eR={'mind':-10,'bonds':-5}),
    c('equipment_war','Lab Drama', eL={'bonds':-15,'mind':10,'research':5}, eR={'mind':-5,'bonds':5}),
    c('shared_desk_drama','Lab Drama', eL={'bonds':-10,'mind':5}, eR={'mind':5,'bonds':-5}),
    c('lab_music_war','Lab Drama', eL={'bonds':-5,'mind':10}, eR={'mind':-5,'body':-5}),
    c('slack_passive_aggressive','Lab Drama', eL={'body':-5,'mind':5,'bonds':5}, eR={'mind':-5}),
    c('excluded_hangout','Social Drama', eL={'bonds':10,'mind':5,'wallet':-5}, eR={'mind':-15,'bonds':-10}),
    c('friend_group_shift','Social Drama', eL={'bonds':10,'mind':-5}, eR={'bonds':5,'mind':5}),
    c('party_before_deadline','Social Drama', eL={'bonds':15,'mind':10,'research':-10,'body':-5}, eR={'research':5,'mind':-5,'bonds':-10}),
    c('relationship_dept','Social Drama', eL={'bonds':5,'mind':-5}, eR={'mind':-5,'bonds':-5}),
    c('the_vent_session','Social Drama', eL={'bonds':15,'mind':-10,'body':-5}, eR={'bonds':-5,'mind':10}),
    c('friend_too_drunk','Social Drama', eL={'bonds':5,'mind':-10}, eR={'bonds':-10,'mind':-5}),
    c('not_invited_grind','Social Drama', eL={'research':10,'mind':-5,'bonds':-10}, eR={'bonds':5,'mind':5}),
    c('your_party_fallout','Social Drama', eL={'bonds':5,'mind':-10}, eR={'bonds':-10,'mind':-5}),
    c('useless_mentor_postdoc','Lab Drama', eL={'bonds':-10,'mind':10}, eR={'mind':-10,'research':5,'body':-5}),
    c('messy_postdoc','Lab Drama', eL={'body':-5,'mind':-10,'bonds':5}, eR={'bonds':-15,'mind':5}),
    c('mislabel_postdoc','Lab Drama', eL={'bonds':10,'research':-5,'mind':-5}, eR={'mind':-5,'research':5}),
    c('docker_refusal_postdoc','Lab Drama', eL={'bonds':-10,'mind':10,'research':5}, eR={'mind':-15,'body':-5,'research':-5}),
    c('phone_postdoc','Lab Drama', eL={'bonds':-10,'mind':5}, eR={'mind':-5,'body':-5,'research':-5}),
    c('slacking_technician','Lab Drama', eL={'body':-10,'mind':-10,'bonds':5}, eR={'bonds':-10,'mind':5}),
    c('plant_path_collab','Collaboration', eL={'bonds':10,'research':-5,'body':-10,'mind':-5}, eR={'bonds':-5,'mind':5}),
    c('no_credit_collab','Collaboration', eL={'bonds':-15,'mind':10,'research':5}, eR={'mind':-15,'bonds':5}),
    c('collab_exploit_pattern','Collaboration', eL={'bonds':-10,'mind':15,'research':5,'network':10}, eR={'bonds':10,'research':-10,'mind':-10}),
    c('teaching_them_R','Collaboration', eL={'bonds':15,'research':-10,'body':-5}, eR={'bonds':-5,'mind':5}),
    c('pi_ethics_scandal','Black Swan', eL={'bonds':15,'mind':-10}, eR={'mind':-5,'bonds':-5}),
    c('class_realization','Privilege', eL={'bonds':5,'mind':-15,'network':5}, eR={'mind':-5,'bonds':-5,'network':-5}),
    c('gacrc_workshop','Pipeline', eL={'research':10,'body':-5,'mind':5}, eR={'mind':-5}),
    c('hpc_no_ai','Pipeline Failure', eL={'mind':5,'bonds':5}, eR={'mind':-5,'research':-5}),
]

PHASE2_CARDS.extend(NEW_PHASE2_CARDS)

# ── PHASE 3 CARDS (21) ──
PHASE3_CARDS = [
    c('when_graduating','Existential Event', minSem=7, eL={'mind':-15,'bonds':-5}, eR={'mind':-5,'bonds':5}),
    c('5th_year_blues','Existential Event', minSem=7, eL={'mind':20,'bonds':10,'wallet':-5,'research':-5}, eR={'mind':5,'body':-5,'research':5}),
    c('chapter_done','Dissertation', minSem=7, eL={'mind':20,'body':15,'bonds':10,'research':-5}, eR={'mind':-5,'wallet':10,'research':10}),
    c('job_market','Career', minSem=8, eL={'wallet':10,'bonds':10,'mind':-15,'network':15}, eR={'mind':-5,'wallet':-5,'research':10}),
    c('industry_vs_academia','Career', minSem=8, eL={'wallet':15,'mind':10,'bonds':-15,'research':-5,'network':5}, eR={'bonds':15,'wallet':-10,'mind':-5,'research':5}),
    c('committee_meeting','Milestone', eL={'mind':-20,'bonds':15,'research':5}, eR={'bonds':10,'mind':-10}),
    c('final_figure','Dissertation', eL={'mind':-15,'bonds':10,'research':10}, eR={'mind':5,'bonds':-10,'research':5}),
    c('last_experiment','Pipeline', eL={'mind':-20,'body':-15,'bonds':10,'research':10}, eR={'bonds':-10,'mind':10}),
    c('writing_hell','Dissertation', minSem=7, eL={'mind':-20,'body':-10,'research':5}, eR={'mind':10,'body':10,'research':-5}),
    c('defense_scheduled_card','Milestone', minSem=9, eL={'mind':-20,'body':-15,'bonds':-15,'research':10}, eR={'bonds':-15,'mind':5}, sets=['defense_set']),
    c('the_submission','Milestone', minSem=9, eL={'mind':20,'bonds':10,'research':10,'network':5}, eR={'mind':5,'bonds':5,'research':5}),
    c('night_before_defense','Milestone', minSem=10, eL={'mind':-10,'bonds':5,'research':5}, eR={'mind':15,'body':10}),
    c('job_talk_brand','Marketing', minSem=8, eL={'bonds':10,'mind':-10,'body':-10,'research':5,'network':10}, eR={'mind':-5,'bonds':-5,'network':5}),
    c('tweetorial','Marketing', minSem=7, eL={'bonds':10,'mind':-15,'wallet':5,'research':5,'network':10}, eR={'mind':10,'bonds':-15}),
    c('safety_net','Privilege', minSem=7, eL={'wallet':5,'mind':-10}, eR={'mind':-15,'wallet':-10}),
    c('who_gets_postdoc','Privilege', minSem=8, eL={'bonds':5,'mind':-15,'network':10}, eR={'mind':-10,'wallet':-5,'body':-5,'network':5}),
    c('ice_storm','Black Swan', minSem=9, eL={'mind':15,'body':10,'research':5}, eR={'mind':-10,'body':5}),
    c('advisor_leaves','Black Swan', minSem=5, eL={'wallet':-8,'bonds':-10,'mind':-5,'research':-5}, eR={'bonds':-20,'mind':-15,'wallet':5,'research':-10}),
    c('committee_betrayal','Cruel', minSem=7, eL={'bonds':-15,'mind':-10,'research':-5}, eR={'bonds':-20,'mind':-5,'research':-5}),
    c('journal_rejection','Cruel', minSem=6, eL={'mind':-15,'bonds':-5,'research':5}, eR={'mind':-20,'body':-10,'research':10}),
    c('advisor_conflict_of_interest','Cruel', minSem=7, eL={'bonds':-20,'mind':5,'research':-5}, eR={'mind':-20,'bonds':-5,'research':-5}),
    c('practice_defense','Preparation', minSem=9, eL={'mind':-15,'bonds':10,'research':5}, eR={'mind':5,'bonds':5,'research':5}),
]

# ── UNIVERSAL CARDS (13 inline + 33 from js file = 46) ──
UNIVERSAL_CARDS = [
    c('rent_hike','Athens Reality', eL={'wallet':-5,'mind':-15,'body':5}, eR={'wallet':-10,'body':-10,'mind':-10}),
    c('health_insurance','Funding', eL={'wallet':5,'mind':-10,'body':-5}, eR={'wallet':-8,'body':-10,'mind':-5}),
    c('car_trouble','Life Event', eL={'wallet':5,'body':5,'mind':-15}, eR={'wallet':-10,'mind':-5}),
    c('good_coffee','Small Win', eL={'mind':15,'body':10,'bonds':5,'research':-5}, eR={'mind':10,'wallet':5,'research':5}),
    c('email_from_stranger','Validation', eL={'bonds':10,'mind':15,'research':5,'network':5}, eR={'mind':20,'research':5}),
    c('power_outage','Athens Life', eL={'mind':-10,'body':-5,'research':5}, eR={'mind':15,'body':10,'research':-5}),
    c('tax_season','Funding', eL={'mind':-10,'wallet':-5}, eR={'mind':-15,'wallet':5}),
    c('adopt_plant','Self Care', eL={'mind':15,'wallet':-5,'bonds':5}, eR={'mind':5,'research':5}),
    c('espresso','Big Mood', eL={'mind':20,'bonds':20,'body':-5,'network':5}, eR={'mind':15,'body':-5}),
    c('free_food','Survival', eL={'bonds':10,'wallet':5,'network':10}, eR={'wallet':10,'body':5,'bonds':-5}),
    c('parking_pass','Athens Reality', eL={'bonds':10,'mind':5,'body':-10}, eR={'wallet':-8,'body':5}),
]

# New universal cards from js file
NEW_UNIVERSAL_CARDS = [
    c('grad_coordinator_saves','Support Staff', eL={'bonds':10,'mind':10}, eR={'bonds':15,'mind':10,'wallet':-3}),
    c('department_staff_party','Support Staff', eL={'bonds':15,'mind':10}, eR={'bonds':10,'mind':5,'wallet':-3}),
    c('it_support_hero','Support Staff', eL={'bonds':10,'mind':15,'research':5}, eR={'mind':10,'research':5}),
    c('staff_coffee_hour','Support Staff', eL={'mind':15,'bonds':10}, eR={'mind':5,'bonds':5}),
    c('ramsey_student_center','Athens Life', eL={'body':15,'research':-5}, eR={'mind':5,'research':5}),
    c('athens_music_scene','Athens Life', eL={'bonds':15,'mind':15,'body':-10,'research':-5}, eR={'mind':-10,'bonds':-10,'research':5}),
    c('watt_street_coffee','Athens Life', eL={'mind':10,'bonds':5}, eR={'research':5,'bonds':-5}),
    c('uga_football_saturday','Athens Life', eL={'bonds':15,'mind':10,'body':5,'research':-10}, eR={'research':10,'mind':-5,'bonds':-10}),
    c('normaltown_run','Self Care', eL={'body':15,'mind':10,'research':-5}, eR={'body':5,'mind':5}),
    c('jittery_joes','Athens Life', eL={'research':10,'body':-10,'mind':5}, eR={'body':5,'mind':5}),
    c('five_points_bar','Athens Life', eL={'bonds':20,'body':-10,'wallet':-5,'mind':10}, eR={'bonds':5,'mind':5,'wallet':-3}),
    c('spring_in_athens','Athens Life', eL={'mind':10,'body':5}, eR={'mind':-5}),
    c('hatch_hackathon','Social', eL={'bonds':20,'mind':15,'research':10,'body':-20}, eR={'bonds':10,'mind':5,'body':5}),
    c('zoo_trip','Social', eL={'bonds':15,'mind':15,'body':5,'research':-5}, eR={'research':5,'mind':-10,'bonds':-10}),
    c('strawberry_picking','Social', eL={'bonds':15,'body':10,'mind':10,'research':-5}, eR={'bonds':5,'body':5,'research':5}),
    c('consulting_club','Networking', eL={'network':10,'mind':-10,'bonds':-5}, eR={'mind':5,'bonds':5}),
    c('linkedin_lunatic','Lab Life', eL={'network':15,'bonds':-10,'mind':-5}, eR={'bonds':5,'mind':5}),
    c('brewery_night','Athens Life', eL={'bonds':20,'body':-15,'wallet':-8,'mind':10}, eR={'bonds':10,'body':-5,'wallet':-3,'mind':5}),
    c('sips_cowork','Athens Life', eL={'network':15,'mind':-10,'research':-5}, eR={'research':10,'mind':5}),
    c('wine_weekend','Athens Life', eL={'network':15,'bonds':10,'wallet':-10,'mind':10}, eR={'mind':15,'body':-5,'wallet':-5,'bonds':5}),
    c('pulled_over','Athens Life', eL={'wallet':-10,'mind':-10}, eR={'wallet':-5,'mind':-5,'body':-5}),
]
UNIVERSAL_CARDS.extend(NEW_UNIVERSAL_CARDS)

# ── CALLBACK CARDS ──
CALLBACK_CARDS = [
    c('student_returns','Callback', eL={'mind':15,'bonds':10,'research':5,'network':5}, eR={'mind':5,'bonds':10,'research':5}, requires=['mentored_student']),
    c('grant_result','Callback', eL={'wallet':20,'mind':15,'bonds':10,'research':10}, eR={'wallet':20,'mind':10,'bonds':10,'research':10}, requires=['applied_grant']),
    c('pipeline_callback','Callback', eL={'mind':15,'bonds':10,'research':10}, eR={'mind':10,'bonds':5,'research':5}, requires=['built_pipeline']),
    c('cookie_callback','Callback', eL={'bonds':20,'mind':10,'wallet':-5,'research':-5}, eR={'mind':10,'bonds':5}, requires=['baked_cookies']),
    c('allnighter_cost','Callback', eL={'body':10,'mind':5,'wallet':-5}, eR={'body':-15,'mind':-10}, requires=['pulled_allnighter']),
    c('quals_callback','Callback', eL={'mind':15,'bonds':10,'research':5}, eR={'mind':10,'bonds':15,'research':5}, requires=['quals_scheduled']),
]

# ── EXCLUSIVE CARDS (archetype) ──
EXCLUSIVE_CARDS = [
    c('volunteered_committee', 'Overachiever', exclusive='overachiever', eL={'bonds':10,'mind':-10,'research':5}, eR={'mind':5,'bonds':-5}),
    c('color_coded_calendar', 'Overachiever', exclusive='overachiever', eL={'bonds':10,'mind':-5}, eR={'mind':5,'research':5}),
    c('overachiever_burnout', 'Overachiever', exclusive='overachiever', eL={'body':10,'mind':5,'research':-5}, eR={'research':10,'body':-15,'mind':-5}),
    c('ai_methods', 'Vibe-Coder', exclusive='vibe_coder', eL={'research':10,'mind':-10,'bonds':-5}, eR={'mind':5,'research':5,'body':-5}),
    c('cursor_incident', 'Vibe-Coder', exclusive='vibe_coder', eL={'bonds':-5,'mind':-10}, eR={'mind':-5,'bonds':-5}),
    c('ai_detector', 'Vibe-Coder', exclusive='vibe_coder', eL={'mind':-15,'bonds':5}, eR={'mind':-5,'research':-5,'body':-5}),
    c('pre_defense_karaoke', 'Fun Haver', exclusive='fun_haver', eL={'bonds':20,'mind':10,'body':-10,'research':-10}, eR={'mind':5,'research':5,'bonds':-10}),
    c('legendary_party', 'Fun Haver', exclusive='fun_haver', eL={'bonds':20,'mind':10,'wallet':-10}, eR={'bonds':-5,'mind':5}),
    c('fun_haver_reality', 'Fun Haver', exclusive='fun_haver', eL={'bonds':-10,'mind':10,'research':5}, eR={'bonds':10,'mind':5,'research':-10}),
    c('visa_renewal', 'Global Student', exclusive='global_student', eL={'bonds':-5,'mind':-10}, eR={'bonds':5,'mind':-5,'wallet':-5}),
    c('community_potluck', 'Global Student', exclusive='global_student', eL={'bonds':20,'body':-10,'wallet':-5,'mind':15}, eR={'bonds':10,'mind':10}),
    c('opt_clock', 'Global Student', exclusive='global_student', minSem=8, eL={'network':15,'mind':-15,'bonds':5}, eR={'mind':-5,'research':5}),
    c('cultural_misunderstanding', 'Global Student', exclusive='global_student', eL={'bonds':5,'mind':-10}, eR={'mind':-5,'bonds':-5}),
    c('homesick_holiday', 'Global Student', exclusive='global_student', eL={'bonds':10,'body':-10,'mind':5}, eR={'mind':-10,'bonds':-5}),
    c('py_in_word', 'Biologist', exclusive='biologist', eL={'research':10,'mind':-10}, eR={'bonds':5,'research':5,'mind':-5}),
    c('send_my_data', 'Biologist', exclusive='biologist', eL={'bonds':-5,'research':10,'mind':5}, eR={'research':5,'mind':-10,'body':-5}),
    c('biologist_committee', 'Biologist', exclusive='biologist', eL={'research':10,'mind':-15,'body':-10}, eR={'mind':-5,'research':-5}),
    c('western_and_rnaseq', 'Double Agent', exclusive='double_agent', eL={'body':-10,'research':5,'mind':-5}, eR={'research':10,'mind':-5,'body':-5}),
    c('which_meeting', 'Double Agent', exclusive='double_agent', eL={'bonds':-10,'mind':-5}, eR={'bonds':-5,'mind':5,'research':5}),
    c('leg_day_vs_lab', 'Gym Bro', exclusive='gym_bro', eL={'body':-10,'mind':-10,'research':5,'bonds':5}, eR={'body':15,'bonds':-10,'mind':5}),
    c('protein_budget', 'Gym Bro', exclusive='gym_bro', eL={'body':-10,'wallet':5,'mind':-5}, eR={'body':5,'wallet':-5,'mind':-5}),
    c('medication_refill', 'Neurodivergent', exclusive='neurodivergent', eL={'mind':15,'body':5,'wallet':-5}, eR={'mind':-20,'research':-10,'body':-5}),
    c('hyperfocus_zone', 'Neurodivergent', exclusive='neurodivergent', eL={'research':20,'body':-15,'bonds':-10}, eR={'body':10,'mind':5,'research':5}),
    c('sensory_overload', 'Neurodivergent', exclusive='neurodivergent', eL={'mind':10,'bonds':-5,'research':5}, eR={'mind':-15,'body':-5}),
]

# ── PI EXCLUSIVE CARDS ──
PI_EXCLUSIVE_CARDS = [
    c('daily_standup','Advisor', exclusive='micromanager', eL={'research':10,'mind':-10,'bonds':5}, eR={'mind':5,'bonds':-10,'research':5}),
    c('weekend_checkin','Advisor', exclusive='micromanager', eL={'body':-10,'research':10,'mind':-10}, eR={'mind':5,'bonds':-15}),
    c('pi_rewrote_abstract','Advisor', exclusive='micromanager', eL={'mind':-10,'research':5,'bonds':5}, eR={'mind':5,'bonds':-10}),
    c('havent_seen_advisor','Advisor', exclusive='ghost', eL={'mind':10,'research':-5,'bonds':-5}, eR={'bonds':5,'mind':-5,'research':5}),
    c('email_void','Advisor', exclusive='ghost', eL={'bonds':-10,'mind':5,'research':5}, eR={'mind':-5,'research':5,'bonds':-5}),
    c('self_advise','Advisor', exclusive='ghost', eL={'mind':10,'research':10,'bonds':-10}, eR={'bonds':5,'mind':-5}),
    c('pi_stole_talk','Advisor', exclusive='exploiter', eL={'bonds':-20,'mind':10,'research':5}, eR={'mind':-15,'bonds':-5,'research':5}),
    c('onedrive_full','Advisor', exclusive='exploiter', eL={'bonds':-10,'mind':-5}, eR={'mind':-5,'research':-5,'body':-5}),
    c('first_draft_author','Advisor', exclusive='exploiter', eL={'bonds':-15,'mind':10}, eR={'mind':-15,'bonds':5,'research':5}),
    c('mental_health_checkin','Advisor', exclusive='mentor', eL={'mind':15,'bonds':15}, eR={'mind':5,'bonds':5}),
    c('pi_advocates_raise','Advisor', exclusive='mentor', eL={'bonds':10,'wallet':10,'mind':10}, eR={'wallet':5,'mind':5}),
    c('gentle_nudge','Advisor', exclusive='mentor', eL={'research':10,'mind':5,'bonds':10}, eR={'mind':-10,'research':5}),
    c('pi_also_panicking','Advisor', exclusive='new_pi', eL={'bonds':15,'mind':-5,'research':5}, eR={'mind':5,'bonds':-5}),
    c('lab_meeting_two','Advisor', exclusive='new_pi', eL={'research':10,'bonds':10,'mind':-5,'body':-5}, eR={'bonds':-5,'mind':-5}),
    c('tenure_review','Advisor', exclusive='new_pi', eL={'research':10,'body':-10,'mind':-10,'bonds':5}, eR={'mind':5,'bonds':-10}),
    c('project_reassigned','Advisor', exclusive='dynasty', eL={'bonds':-15,'mind':10,'research':5}, eR={'mind':-15,'research':-10,'bonds':5}),
    c('pi_forgot_name','Advisor', exclusive='dynasty', eL={'bonds':-10,'mind':5}, eR={'mind':-10,'bonds':-5}),
    c('excellent_funding','Advisor', exclusive='dynasty', eL={'wallet':10,'research':10,'network':10,'bonds':-5}, eR={'bonds':10,'wallet':-10,'mind':5}),
]

# ── MILESTONE CARDS ──
MILESTONE_CARDS = [
    c('ms_rotation','MILESTONE', milestone=True, semester=1, piSelection=True),
    c('ms_quals','MILESTONE', milestone=True, semester=4, eL={'body':-15,'mind':10}, eR={'mind':15}, sets=['quals_passed']),
    c('ms_committee_1','MILESTONE', milestone=True, semester=6, eL={'bonds':10,'mind':5}, eR={'bonds':-5,'mind':-10}, sets=['committee_1_passed']),
    c('ms_committee_2','MILESTONE', milestone=True, semester=8, eL={'bonds':10,'mind':5}, eR={'bonds':-5,'mind':-15}, sets=['committee_2_passed']),
    c('ms_defense_sched','MILESTONE', milestone=True, semester=9, eL={'mind':-15,'bonds':5}, eR=None, sets=['defense_scheduled']),
    c('ms_defense','MILESTONE', milestone=True, semester=10, eL={'mind':20,'bonds':10,'research':5}, eR={'mind':20,'bonds':10}, sets=['defended']),
]

print(f"Cards loaded: P1={len(PHASE1_CARDS)} P2={len(PHASE2_CARDS)} P3={len(PHASE3_CARDS)} "
      f"Univ={len(UNIVERSAL_CARDS)} CB={len(CALLBACK_CARDS)} Excl={len(EXCLUSIVE_CARDS)} "
      f"PI_Excl={len(PI_EXCLUSIVE_CARDS)} Mile={len(MILESTONE_CARDS)}")
print(f"Total: {len(PHASE1_CARDS)+len(PHASE2_CARDS)+len(PHASE3_CARDS)+len(UNIVERSAL_CARDS)+len(CALLBACK_CARDS)+len(EXCLUSIVE_CARDS)+len(PI_EXCLUSIVE_CARDS)+len(MILESTONE_CARDS)}")

## 2. Game Engine

In [ ]:
def get_phase(semester):
    if semester <= 2: return 1
    if semester <= 6: return 2
    return 3

def apply_perk(arch, stat, delta, tag, bonds):
    """Archetype perk modifier."""
    tech_tags = ['Pipeline Failure', 'Pipeline']
    social_tags = ['Social', 'Human Existence', 'Athens Life', 'Social Drama', 'Lab Drama']
    if arch == 'vibe_coder':
        if tag in tech_tags and delta > 0: return math.floor(delta * 1.5)
        if tag in social_tags and delta < 0: return math.floor(delta * 1.5)
    elif arch == 'fun_haver':
        if stat == 'bonds' and delta > 0: return math.floor(delta * 1.5)
        if stat == 'mind' and delta < 0: return math.floor(delta * 1.3)
        if stat == 'research' and delta > 0: return math.ceil(delta / 2)
    elif arch == 'global_student':
        if stat == 'bonds' and delta < 0: return math.ceil(delta / 2)
    elif arch == 'biologist':
        if stat == 'research' and delta < 0 and tag in tech_tags:
            if bonds > 50: return math.ceil(delta / 2)
            if bonds < 30: return math.floor(delta * 2)
        if stat == 'research' and delta > 0 and tag in ('Lab Life', 'Lab Politics'):
            return math.floor(delta * 1.3)
    elif arch == 'double_agent':
        if stat == 'research' and delta > 0: return delta + 3
    elif arch == 'gym_bro':
        if stat == 'wallet' and delta < 0: return math.floor(delta * 1.3)
        if stat == 'mind' and delta < 0: return math.floor(delta * 1.3)
    elif arch == 'neurodivergent':
        if stat == 'mind': return math.floor(delta * 1.5)
        if stat == 'research': return math.floor(delta * 1.3)
    return delta

def apply_pi_perk(pi_type, stat, delta, tag):
    """PI perk modifier."""
    is_advisor = (tag == 'Advisor')
    if pi_type == 'micromanager':
        if stat == 'research' and delta > 0: return math.floor(delta * 1.2)
        if stat == 'mind' and delta < 0 and is_advisor: return math.floor(delta * 1.3)
    elif pi_type == 'ghost':
        if stat == 'research' and delta > 0: return math.floor(delta * 0.8)
    elif pi_type == 'exploiter':
        if stat == 'research' and delta > 0: return math.floor(delta * 1.3)
        if stat == 'wallet' and delta < 0: return math.floor(delta * 1.3)
        if stat == 'bonds' and delta < 0 and is_advisor: return math.floor(delta * 1.3)
    elif pi_type == 'mentor':
        if delta < 0 and is_advisor: return min(delta + 3, 0)
        if stat == 'bonds' and delta > 0 and is_advisor: return math.floor(delta * 1.2)
        if stat == 'research' and delta > 0: return min(delta, 10)
    elif pi_type == 'new_pi':
        if stat == 'research' and delta > 0: return math.floor(delta * 1.25)
        if stat == 'mind' and delta < 0: return math.floor(delta * 1.2)
    elif pi_type == 'dynasty':
        if stat == 'wallet' and delta > 0: return math.floor(delta * 1.2)
        if stat == 'bonds' and delta < 0: return math.floor(delta * 1.3)
    return delta

def clamp(v, lo=0, hi=100):
    return max(lo, min(hi, v))

In [ ]:
STRATEGIES = {
    'random':     lambda card, gs: random.choice(['left','right']),
    'left_bias':  lambda card, gs: 'left' if random.random() < 0.65 else 'right',
    'right_bias': lambda card, gs: 'right' if random.random() < 0.65 else 'left',
    'greedy_mind': lambda card, gs: _greedy_pick(card, gs, 'mind'),
    'greedy_research': lambda card, gs: _greedy_pick(card, gs, 'research'),
    'balanced':   lambda card, gs: _balanced_pick(card, gs),
}

def _sum_effects(effects):
    if not effects: return 0
    return sum(v for k, v in effects.items() if k != 'network')

def _greedy_pick(card, gs, target_stat):
    eL = card['eL'] or {}
    eR = card['eR'] or {}
    lv = eL.get(target_stat, 0)
    rv = eR.get(target_stat, 0)
    if lv > rv: return 'left'
    if rv > lv: return 'right'
    return 'left' if _sum_effects(eL) >= _sum_effects(eR) else 'right'

def _balanced_pick(card, gs):
    """Pick the side that helps the weakest stat most."""
    weakest = min(gs['st'], key=gs['st'].get)
    eL = card['eL'] or {}
    eR = card['eR'] or {}
    lv = eL.get(weakest, 0)
    rv = eR.get(weakest, 0)
    if lv > rv: return 'left'
    if rv > lv: return 'right'
    return 'left' if _sum_effects(eL) >= _sum_effects(eR) else 'right'

In [ ]:
def simulate_game(archetype, pi_type, strategy='random', rng=None):
    """Run one full game. Returns dict with ending, semester, final stats, cards played."""
    if rng is None:
        rng = random.Random()
    
    st = dict(ARCHETYPE_DATA[archetype]['st'])
    semester = 1
    card_count = 0  # cards in current semester
    total_cards = 0
    network = 0
    quals_attempts = 0
    memory = set()
    strategy_fn = STRATEGIES[strategy] if isinstance(strategy, str) else strategy
    
    stat_history = []  # track stats over time
    
    def get_milestone(sem):
        for m in MILESTONE_CARDS:
            if m['semester'] == sem: return m
        return None
    
    next_milestone = get_milestone(semester)
    
    def draw_card():
        nonlocal next_milestone, card_count, semester
        phase = get_phase(semester)
        
        # Serve milestone after 3 cards
        if card_count >= 3 and next_milestone:
            ms = next_milestone
            next_milestone = None
            if ms['piSelection']:
                return 'PI_SELECTION'
            # Committee research checks
            if ms['id'] == 'ms_committee_1' and st['research'] < 25:
                st['mind'] = max(0, st['mind'] - 10)
            if ms['id'] == 'ms_committee_2' and st['research'] < 35:
                st['mind'] = max(0, st['mind'] - 15)
            return ms
        
        if card_count >= 3 and not next_milestone:
            card_count = 0
        
        # Build pool
        if phase == 1: pool = list(PHASE1_CARDS)
        elif phase == 2: pool = list(PHASE2_CARDS)
        else: pool = list(PHASE3_CARDS)
        
        pool.extend(UNIVERSAL_CARDS)
        
        # Filter by semester and seen
        pool = [cd for cd in pool
                if (cd['minSem'] is None or semester >= cd['minSem'])
                and (cd['maxSem'] is None or semester <= cd['maxSem'])
                and cd['id'] not in memory]
        
        # Callback cards
        for cb in CALLBACK_CARDS:
            if cb['id'] in memory: continue
            if cb['requires'] and all(r in memory for r in cb['requires']):
                pool.append(cb)
        
        # Archetype exclusive (2x weight)
        for ex in EXCLUSIVE_CARDS:
            if ex['exclusive'] != archetype: continue
            if ex['id'] in memory: continue
            if ex['minSem'] and semester < ex['minSem']: continue
            if ex['maxSem'] and semester > ex['maxSem']: continue
            pool.append(ex)
            pool.append(ex)  # 2x
        
        # PI exclusive (2x weight)
        if pi_type:
            for px in PI_EXCLUSIVE_CARDS:
                if px['exclusive'] != pi_type: continue
                if px['id'] in memory: continue
                if px['minSem'] and semester < px['minSem']: continue
                if px['maxSem'] and semester > px['maxSem']: continue
                pool.append(px)
                pool.append(px)  # 2x
        
        if not pool:
            pool = [cd for cd in UNIVERSAL_CARDS if cd['id'] not in memory]
        if not pool:
            return None
        
        return rng.choice(pool)
    
    # ── Main loop ──
    max_iterations = 500  # safety
    iterations = 0
    gs = {'st': st, 'semester': semester, 'archetype': archetype, 'pi_type': pi_type}
    
    while iterations < max_iterations:
        iterations += 1
        card = draw_card()
        
        if card is None:
            return {'ending': 'defended', 'semester': semester, 'st': dict(st),
                    'network': network, 'total_cards': total_cards, 'history': stat_history}
        
        if card == 'PI_SELECTION':
            # PI already set from function args
            card_count = 0
            semester += 1
            next_milestone = get_milestone(semester)
            continue
        
        # Pick side
        gs['st'] = st
        gs['semester'] = semester
        side = strategy_fn(card, gs)
        effects = card['eL'] if side == 'left' else card['eR']
        
        # ── Defense delay ──
        if card['milestone'] and side == 'right' and card['id'] == 'ms_defense_sched':
            if 'defense_delayed' in memory:
                return {'ending': 'mastered_out', 'cause': 'Delayed Too Long',
                        'semester': semester, 'st': dict(st),
                        'network': network, 'total_cards': total_cards, 'history': stat_history}
            memory.add('defense_delayed')
            # null effects → advance semester
            card_count += 1
            total_cards += 1
            semester += 1
            next_milestone = get_milestone(semester)
            stat_history.append(dict(st))
            continue
        
        # ── Quals failure ──
        if card['id'] == 'ms_quals':
            threshold = 30 if archetype == 'biologist' else 25
            if st['research'] < threshold:
                quals_attempts += 1
                if quals_attempts >= 3:
                    return {'ending': 'mastered_out', 'cause': 'Failed Quals (3x)',
                            'semester': semester, 'st': dict(st),
                            'network': network, 'total_cards': total_cards, 'history': stat_history}
                st['mind'] = max(0, st['mind'] - 15)
                memory.add(f'quals_retry_{quals_attempts}')
                card_count += 1
                total_cards += 1
                semester += 1
                next_milestone = get_milestone(4)  # re-queue quals
                stat_history.append(dict(st))
                continue
        
        # ── Defense research gate ──
        if card['id'] == 'ms_defense' and st['research'] < 30:
            return {'ending': 'mastered_out', 'cause': 'Insufficient Research',
                    'semester': semester, 'st': dict(st),
                    'network': network, 'total_cards': total_cards, 'history': stat_history}
        
        # ── Apply effects ──
        if effects is None:
            card_count += 1
            total_cards += 1
            semester += 1
            next_milestone = get_milestone(semester)
            stat_history.append(dict(st))
            continue
        
        for stat_key, delta in effects.items():
            if stat_key == 'network':
                network = max(0, network + delta)
                continue
            d = apply_perk(archetype, stat_key, delta, card['tag'], st.get('bonds', 50))
            if pi_type:
                d = apply_pi_perk(pi_type, stat_key, d, card['tag'])
            st[stat_key] = clamp(st[stat_key] + d)
        
        # ── Post-choice archetype effects ──
        if archetype == 'overachiever':
            weak = [s for s, v in st.items() if v < 20 and s != 'research']
            if weak:
                pick = rng.choice(weak)
                st[pick] = min(100, st[pick] + 5)
        if archetype == 'gym_bro':
            if st['body'] < 25: st['body'] = 25
        if archetype == 'double_agent':
            drain = rng.choice(['mind','body','wallet','bonds'])
            st[drain] = max(0, st[drain] - 2)
        
        # ── Passive drains ──
        if st['wallet'] < 5: st['wallet'] = 5
        if st['wallet'] < 20:
            st['mind'] = max(0, st['mind'] - 3)
            st['body'] = max(0, st['body'] - 2)
        if st['bonds'] < 20:
            st['mind'] = max(0, st['mind'] - 2)
            st['body'] = max(0, st['body'] - 1)
        if st['research'] < 20 and semester >= 3:
            st['mind'] = max(0, st['mind'] - 2)
        if st['research'] < 15 and semester >= 5:
            st['bonds'] = max(0, st['bonds'] - 2)
        if archetype == 'global_student' and semester >= 8:
            st['mind'] = max(0, st['mind'] - 3)
        
        # ── Flags ──
        if card.get('sets'):
            for flag in card['sets']:
                memory.add(flag)
        memory.add(card['id'])
        
        total_cards += 1
        card_count += 1
        stat_history.append(dict(st))
        
        # ── Check endings ──
        if st['wallet'] <= 0:
            return {'ending':'broke','cause':'Wallet','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        if st['body'] <= 0:
            return {'ending':'hospitalized','cause':'Body','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        if st['mind'] <= 0:
            return {'ending':'burnt_out','cause':'Mind','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        if st['bonds'] <= 0:
            return {'ending':'disappeared','cause':'Bonds','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        if 'defended' in memory:
            return {'ending':'defended','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        if semester > 10 and 'defended' not in memory:
            return {'ending':'mastered_out','cause':'Time','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}
        
        # ── Advance semester ──
        if card_count >= 3 and not next_milestone:
            semester += 1
            next_milestone = get_milestone(semester)
    
    return {'ending':'mastered_out','cause':'Timeout','semester':semester,'st':dict(st),'network':network,'total_cards':total_cards,'history':stat_history}

# Quick smoke test
res = simulate_game('overachiever', 'mentor', 'random')
print(f"Smoke test: {res['ending']} in sem {res['semester']}, {res['total_cards']} cards")

## 3. Run Simulations

In [ ]:
N_RUNS = 2000  # per combo
STRATEGY = 'random'

archetypes = list(ARCHETYPE_DATA.keys())
pi_types = PI_TYPES

results = []
for arch in archetypes:
    for pi in pi_types:
        for i in range(N_RUNS):
            r = simulate_game(arch, pi, STRATEGY)
            r['archetype'] = arch
            r['pi_type'] = pi
            r['run'] = i
            results.append(r)

df = pd.DataFrame(results)
df['defended'] = (df['ending'] == 'defended').astype(int)
print(f"Total runs: {len(df)}")
df['ending'].value_counts()

## 4. Overall Ending Distribution

In [ ]:
ending_pct = df['ending'].value_counts(normalize=True).mul(100).round(1)
print("=== Overall Ending Distribution (%) ===")
print(ending_pct.to_string())
print()

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(8, 4))
    order = ['defended','mastered_out','burnt_out','hospitalized','broke','disappeared']
    colors = ['#4CAF50','#FF9800','#9C27B0','#F44336','#795548','#607D8B']
    counts = df['ending'].value_counts().reindex(order, fill_value=0)
    ax.bar(counts.index, counts.values, color=colors)
    ax.set_title('Overall Ending Distribution')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + len(df)*0.005, f'{v/len(df)*100:.1f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 5. Defense Rate by Archetype × PI

In [ ]:
pivot = df.pivot_table(values='defended', index='archetype', columns='pi_type', aggfunc='mean').mul(100).round(1)
pivot['ALL'] = df.groupby('archetype')['defended'].mean().mul(100).round(1)

print("=== Defense Rate (%) by Archetype × PI ===")
print(pivot.to_string())
print()

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(pivot.drop(columns='ALL'), annot=True, fmt='.1f', cmap='RdYlGn',
                vmin=0, vmax=100, ax=ax, linewidths=0.5)
    ax.set_title('Defense Rate (%) — Archetype × PI Type')
    plt.tight_layout()
    plt.show()

## 6. Ending Breakdown by Archetype

In [ ]:
ending_by_arch = df.groupby(['archetype','ending']).size().unstack(fill_value=0)
ending_by_arch_pct = ending_by_arch.div(ending_by_arch.sum(axis=1), axis=0).mul(100).round(1)

order = ['defended','mastered_out','burnt_out','hospitalized','broke','disappeared']
ending_by_arch_pct = ending_by_arch_pct.reindex(columns=order, fill_value=0)

print("=== Ending % by Archetype ===")
print(ending_by_arch_pct.to_string())
print()

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(12, 6))
    ending_by_arch_pct.plot(kind='bar', stacked=True, ax=ax,
                            color=['#4CAF50','#FF9800','#9C27B0','#F44336','#795548','#607D8B'])
    ax.set_title('Ending Distribution by Archetype')
    ax.set_ylabel('%')
    ax.legend(title='Ending', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 7. Ending Breakdown by PI Type

In [ ]:
ending_by_pi = df.groupby(['pi_type','ending']).size().unstack(fill_value=0)
ending_by_pi_pct = ending_by_pi.div(ending_by_pi.sum(axis=1), axis=0).mul(100).round(1)
ending_by_pi_pct = ending_by_pi_pct.reindex(columns=order, fill_value=0)

print("=== Ending % by PI Type ===")
print(ending_by_pi_pct.to_string())
print()

if HAS_PLT:
    fig, ax = plt.subplots(figsize=(10, 6))
    ending_by_pi_pct.plot(kind='bar', stacked=True, ax=ax,
                          color=['#4CAF50','#FF9800','#9C27B0','#F44336','#795548','#607D8B'])
    ax.set_title('Ending Distribution by PI Type')
    ax.set_ylabel('%')
    ax.legend(title='Ending', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 8. Average Survival Semester (for non-defenders)

In [ ]:
non_defended = df[df['ending'] != 'defended']

avg_sem_arch = non_defended.groupby('archetype')['semester'].mean().round(2)
avg_sem_pi = non_defended.groupby('pi_type')['semester'].mean().round(2)

print("=== Avg Death Semester by Archetype (non-defenders) ===")
print(avg_sem_arch.sort_values().to_string())
print()
print("=== Avg Death Semester by PI (non-defenders) ===")
print(avg_sem_pi.sort_values().to_string())

## 9. Average Final Stats (all runs)

In [ ]:
# Extract final stats into columns
for stat in ['mind','body','wallet','bonds','research']:
    df[f'final_{stat}'] = df['st'].apply(lambda x: x.get(stat, 0))

stat_cols = [f'final_{s}' for s in ['mind','body','wallet','bonds','research']]

print("=== Average Final Stats by Archetype ===")
print(df.groupby('archetype')[stat_cols].mean().round(1).to_string())
print()
print("=== Average Final Stats by PI ===")
print(df.groupby('pi_type')[stat_cols].mean().round(1).to_string())

## 10. Strategy Comparison

In [ ]:
N_STRAT = 1000
strat_results = []

for strat_name in STRATEGIES:
    for i in range(N_STRAT):
        arch = random.choice(archetypes)
        pi = random.choice(pi_types)
        r = simulate_game(arch, pi, strat_name)
        strat_results.append({'strategy': strat_name, 'ending': r['ending'],
                              'semester': r['semester'], 'defended': int(r['ending']=='defended')})

sdf = pd.DataFrame(strat_results)

print("=== Defense Rate by Strategy ===")
print(sdf.groupby('strategy')['defended'].mean().mul(100).round(1).sort_values(ascending=False).to_string())
print()
print("=== Ending Distribution by Strategy ===")
strat_endings = sdf.groupby(['strategy','ending']).size().unstack(fill_value=0)
strat_endings_pct = strat_endings.div(strat_endings.sum(axis=1), axis=0).mul(100).round(1)
print(strat_endings_pct.to_string())

## 11. Cause of Death Analysis

In [ ]:
deaths = df[df['ending'] != 'defended'].copy()
death_cause = deaths['ending'].value_counts(normalize=True).mul(100).round(1)

print("=== Non-Defense Endings Distribution ===")
print(death_cause.to_string())
print()

# Which stat kills each archetype most?
print("=== Most Common Death per Archetype ===")
for arch in archetypes:
    sub = deaths[deaths['archetype'] == arch]
    if len(sub) == 0:
        print(f"  {arch}: no deaths!")
        continue
    top = sub['ending'].value_counts().head(3)
    top_str = ', '.join(f"{e} ({v/len(sub)*100:.0f}%)" for e, v in top.items())
    print(f"  {arch}: {top_str}")

## 12. Balance Summary & Recommendations

In [ ]:
print("="*60)
print("BALANCE SUMMARY")
print("="*60)

# Overall
overall_def = df['defended'].mean() * 100
print(f"\nOverall defense rate: {overall_def:.1f}%")
print(f"Target range: 15-35% (challenging but achievable)")
print()

# Best/worst combos
combo = df.groupby(['archetype','pi_type'])['defended'].mean().mul(100).round(1)
print(f"Easiest combo:  {combo.idxmax()} → {combo.max():.1f}%")
print(f"Hardest combo:  {combo.idxmin()} → {combo.min():.1f}%")
print(f"Spread:         {combo.max() - combo.min():.1f} pp")
print()

# Archetype ranking
arch_rate = df.groupby('archetype')['defended'].mean().mul(100).round(1).sort_values(ascending=False)
print("Archetype defense rates (high → low):")
for a, r in arch_rate.items():
    bar = '█' * int(r / 2)
    print(f"  {a:20s} {r:5.1f}% {bar}")

print()
pi_rate = df.groupby('pi_type')['defended'].mean().mul(100).round(1).sort_values(ascending=False)
print("PI defense rates (high → low):")
for p, r in pi_rate.items():
    bar = '█' * int(r / 2)
    print(f"  {p:20s} {r:5.1f}% {bar}")

print()
print("─" * 60)
print("FLAGS:")
if overall_def > 40:
    print("  ⚠️  Game may be too easy overall")
elif overall_def < 10:
    print("  ⚠️  Game may be too hard overall")
else:
    print("  ✅ Overall difficulty looks reasonable")

if combo.max() - combo.min() > 40:
    print("  ⚠️  Large spread between easiest/hardest combo (>40pp)")

for a, r in arch_rate.items():
    if r < 5: print(f"  ⚠️  {a} almost never defends ({r}%)")
    if r > 60: print(f"  ⚠️  {a} defends too easily ({r}%)")

for p, r in pi_rate.items():
    if r < 5: print(f"  ⚠️  {p} PI almost never allows defense ({r}%)")
    if r > 60: print(f"  ⚠️  {p} PI makes defense too easy ({r}%)")